# Prompt Engineering for RAG

**Goal:** Understand how prompt template design affects the quality of RAG-generated answers.

We'll explore the three built-in prompt templates:
1. **default_qa** — Simple context + question format
2. **structured** — Numbered chunks with a "say I don't know" guardrail
3. **chain_of_thought** — Encourages step-by-step reasoning

For each template, we'll examine:
- The rendered prompt structure
- How context chunks are presented to the LLM
- Trade-offs between template styles

In [ ]:
from src.generate.prompt_templates import render_prompt

# Sample retrieved chunks (simulating retriever output)
chunks = [
    "The Great Wall of China is a series of fortifications built along the historical northern borders of China. Construction began in the 7th century BC, with the most well-known sections built during the Ming Dynasty (1368–1644).",
    "The wall stretches approximately 21,196 kilometers (13,171 miles) across northern China. It was built using materials like stone, brick, tamped earth, and wood depending on the region and era.",
    "The Great Wall was designated a UNESCO World Heritage Site in 1987. It attracts millions of visitors annually and is one of the most recognizable landmarks in the world.",
]

question = "How long is the Great Wall of China and when was it built?"

## 1. Default QA Template

The simplest template — presents context chunks sequentially, followed by the question. The LLM receives no special instructions about how to answer.

In [ ]:
default_prompt = render_prompt("default_qa", chunks, question)
print(default_prompt)

**Observations:**
- Chunks are listed one after another with no numbering
- No instructions about what to do when context is insufficient
- Best for: straightforward factual questions where the answer is clearly in the context

## 2. Structured Template

Adds two key improvements: numbered chunk references and a "say I don't know" guardrail instruction.

In [ ]:
structured_prompt = render_prompt("structured", chunks, question)
print(structured_prompt)

**Observations:**
- Each chunk gets a numbered reference `[1]`, `[2]`, `[3]` — makes it easy for the LLM to cite sources
- Explicit instruction: "Answer using ONLY the provided context" — reduces hallucination
- Guardrail: "If the context does not contain enough information, say I don't know"
- Best for: production use where factual accuracy matters more than creativity

## 3. Chain-of-Thought Template

Instructs the LLM to reason through the context step by step before answering.

In [ ]:
cot_prompt = render_prompt("chain_of_thought", chunks, question)
print(cot_prompt)

**Observations:**
- Explicit instruction to "identify relevant facts" then "reason step by step"
- Ends with "Reasoning and Answer:" instead of just "Answer:" — signals the LLM to show its work
- Best for: complex questions that require synthesizing information from multiple chunks

## 4. Comparing Prompt Lengths

Different templates produce different prompt lengths, which affects latency and token costs.

In [ ]:
templates = ["default_qa", "structured", "chain_of_thought"]
for name in templates:
    prompt = render_prompt(name, chunks, question)
    words = len(prompt.split())
    chars = len(prompt)
    print(f"{name:20s}  {words:4d} words  {chars:5d} chars")

## 5. Edge Cases: Empty Context and Missing Answers

How do templates behave when retrieval returns no relevant chunks?

In [ ]:
# What happens with no context chunks?
for name in templates:
    prompt = render_prompt(name, [], "What is quantum computing?")
    print(f"=== {name} (empty context) ===")
    print(prompt)
    print()

**Key insight:** With empty context, the `structured` template still instructs the LLM to say "I don't know" — this is the guardrail in action. The other templates leave it up to the LLM, which may hallucinate an answer from its training data.

## 6. Special Characters and Injection Resistance

RAG prompts must handle real-world text that contains special characters, code, and potentially adversarial content.

In [ ]:
# Test with special characters and code content
special_chunks = [
    "Temperature should be > 100°C & < 200°C for the reaction.",
    "Use the formula: E = mc² where c ≈ 3×10⁸ m/s.",
    "```python\ndef hello():\n    print('Hello, world!')\n```",
]

prompt = render_prompt("structured", special_chunks, "What is the formula?")
print(prompt)
print()

# Verify special characters are preserved (not escaped)
assert ">" in prompt and "<" in prompt and "²" in prompt
print("✓ Special characters preserved correctly")

## 7. Template Selection Guide

| Template | Use When | Pros | Cons |
|---|---|---|---|
| `default_qa` | Quick prototyping, simple factual Q&A | Minimal overhead, lowest token count | No hallucination guardrails |
| `structured` | Production systems, user-facing answers | Numbered refs, "I don't know" guardrail | Slightly more tokens |
| `chain_of_thought` | Complex multi-hop questions, debugging | Shows reasoning, better for synthesis | Longest output, slower |

**Recommendation:** Start with `structured` for most use cases. Switch to `chain_of_thought` when answers require reasoning across multiple chunks. Use `default_qa` only for benchmarking template overhead.

## Next Steps

- **Run with a real LLM** (Ollama) to see how different templates affect answer quality
- **Build a pipeline** with `src/pipeline.py` to test templates end-to-end
- **Track results** with the experiment runner to compare template performance across queries